# JARVIS Basic Personal Assistant (Google Colab)

This notebook builds the safe first stage of a personal assistant:

- SQLite-backed tasks, reminders, and chat history
- One-time and recurring reminders with APScheduler
- Natural-language commands for common task/reminder actions
- Safe system status checks using `psutil`
- A simple widget chat interface

**Colab limitation:** reminders only execute while the Colab runtime is alive. If the runtime disconnects, pauses, or is recycled, APScheduler cannot run jobs. For reliable always-on reminders, move the worker to a persistent server later.

This first version does not require an API key and does not execute shell commands, file deletion, purchases, messages, or other sensitive actions.

In [ ]:
%pip -q install apscheduler==3.10.4 sqlalchemy psutil dateparser ipywidgets

## 1. Imports and configuration

The database files are stored in the current Colab workspace. Download them before ending a session if you need a local backup.

In [ ]:
from __future__ import annotations

import json
import re
import sqlite3
import threading
import uuid
from datetime import datetime, timezone
from pathlib import Path

import dateparser
import psutil
from apscheduler.jobstores.sqlalchemy import SQLAlchemyJobStore
from sqlalchemy.pool import NullPool
from apscheduler.schedulers.background import BackgroundScheduler
from apscheduler.triggers.date import DateTrigger
from apscheduler.triggers.interval import IntervalTrigger

DB_PATH = Path("jarvis.db")
JOB_DB_PATH = Path("jarvis_scheduler.db")
TIMEZONE = "Asia/Kolkata"
MAX_TEXT = 500
_db_lock = threading.Lock()
print(f"Database: {DB_PATH.resolve()}")
print(f"Timezone: {TIMEZONE}")

## 2. SQLite storage

    Tables are intentionally simple and can be migrated later. All writes use parameterized SQL.

In [ ]:
def now_iso():
    return datetime.now(timezone.utc).isoformat()

def connect_db():
    connection = sqlite3.connect(DB_PATH)
    connection.row_factory = sqlite3.Row
    connection.execute("PRAGMA foreign_keys = ON")
    return connection

def initialize_db():
    with connect_db() as db:
        db.executescript("""
        CREATE TABLE IF NOT EXISTS tasks (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            priority TEXT NOT NULL DEFAULT 'medium'
                CHECK(priority IN ('low', 'medium', 'high')),
            due_at TEXT,
            completed INTEGER NOT NULL DEFAULT 0,
            created_at TEXT NOT NULL,
            updated_at TEXT NOT NULL
        );
        CREATE TABLE IF NOT EXISTS reminders (
            id TEXT PRIMARY KEY,
            text TEXT NOT NULL,
            run_at TEXT NOT NULL,
            recurrence TEXT,
            active INTEGER NOT NULL DEFAULT 1,
            last_fired_at TEXT,
            created_at TEXT NOT NULL
        );
        CREATE TABLE IF NOT EXISTS chat_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            role TEXT NOT NULL CHECK(role IN ('user', 'assistant')),
            content TEXT NOT NULL,
            created_at TEXT NOT NULL
        );
        """)

initialize_db()
print("SQLite schema ready")

## 3. Task management

    These functions are the core API used by both tests and the chat interface.

In [ ]:
def _validate_text(value, field="text"):
    value = str(value).strip()
    if not value or len(value) > MAX_TEXT:
        raise ValueError(f"{field} must contain 1-{MAX_TEXT} characters")
    return value

def create_task(title, priority="medium", due_at=None):
    title = _validate_text(title, "Task title")
    if priority not in {"low", "medium", "high"}:
        raise ValueError("Priority must be low, medium, or high")
    timestamp = now_iso()
    with connect_db() as db:
        cursor = db.execute(
            "INSERT INTO tasks(title, priority, due_at, created_at, updated_at) VALUES (?, ?, ?, ?, ?)",
            (title, priority, due_at, timestamp, timestamp),
        )
    return get_task(cursor.lastrowid)

def get_task(task_id):
    with connect_db() as db:
        row = db.execute("SELECT * FROM tasks WHERE id = ?", (int(task_id),)).fetchone()
    return dict(row) if row else None

def list_tasks(include_completed=False):
    query = "SELECT * FROM tasks"
    if not include_completed:
        query += " WHERE completed = 0"
    query += " ORDER BY completed, CASE priority WHEN 'high' THEN 1 WHEN 'medium' THEN 2 ELSE 3 END, id"
    with connect_db() as db:
        return [dict(row) for row in db.execute(query)]

def update_task(task_id, title=None, priority=None, due_at=None):
    task = get_task(task_id)
    if not task:
        raise ValueError("Task not found")
    title = task["title"] if title is None else _validate_text(title, "Task title")
    priority = task["priority"] if priority is None else priority
    if priority not in {"low", "medium", "high"}:
        raise ValueError("Priority must be low, medium, or high")
    with connect_db() as db:
        db.execute("UPDATE tasks SET title = ?, priority = ?, due_at = ?, updated_at = ? WHERE id = ?",
                   (title, priority, due_at if due_at is not None else task["due_at"], now_iso(), int(task_id)))
    return get_task(task_id)

def complete_task(task_id):
    with connect_db() as db:
        result = db.execute("UPDATE tasks SET completed = 1, updated_at = ? WHERE id = ?", (now_iso(), int(task_id)))
    if result.rowcount == 0:
        raise ValueError("Task not found")
    return get_task(task_id)

def delete_task(task_id):
    with connect_db() as db:
        result = db.execute("DELETE FROM tasks WHERE id = ?", (int(task_id),))
    if result.rowcount == 0:
        raise ValueError("Task not found")
    return {"deleted": int(task_id)}

## 4. Reminders and scheduler

Reminder metadata is saved in SQLite and APScheduler jobs are stored in a second SQLite job store. The callback only records delivery in the database and prints a notification; it does not send external messages.

In [ ]:
def _reminder_callback(reminder_id):
    with connect_db() as db:
        row = db.execute("SELECT * FROM reminders WHERE id = ? AND active = 1", (reminder_id,)).fetchone()
        if not row:
            return
        db.execute("UPDATE reminders SET last_fired_at = ? WHERE id = ?", (now_iso(), reminder_id))
    print(f"\n[JARVIS REMINDER] {row['text']} (reminder {reminder_id})")

scheduler = BackgroundScheduler(
    timezone=TIMEZONE,
    jobstores={"default": SQLAlchemyJobStore(url=f"sqlite:///{JOB_DB_PATH}", engine_options={"poolclass": NullPool})},
)

def _parse_run_at(value):
    parsed = dateparser.parse(value, settings={"TIMEZONE": TIMEZONE, "RETURN_AS_TIMEZONE_AWARE": True, "PREFER_DATES_FROM": "future"})
    if parsed is None:
        raise ValueError(f"Could not understand reminder time: {value}")
    return parsed

def create_reminder(text, when, recurrence=None, start_scheduler=True):
    text = _validate_text(text, "Reminder text")
    run_at = _parse_run_at(when)
    reminder_id = uuid.uuid4().hex[:10]
    with connect_db() as db:
        db.execute("INSERT INTO reminders(id, text, run_at, recurrence, created_at) VALUES (?, ?, ?, ?, ?)",
                   (reminder_id, text, run_at.isoformat(), recurrence, now_iso()))
    if recurrence:
        recurrence = recurrence.lower().strip()
        match = re.fullmatch(r"every\s+(\d+)\s+(minute|hour|day|week)s?", recurrence)
        if not match:
            raise ValueError("Recurrence must look like 'every 30 minutes' or 'every 1 day'")
        amount, unit = int(match.group(1)), match.group(2)
        scheduler.add_job(_reminder_callback, IntervalTrigger(**{unit + "s": amount}, timezone=TIMEZONE), args=[reminder_id], id=reminder_id, replace_existing=True)
    else:
        scheduler.add_job(_reminder_callback, DateTrigger(run_date=run_at, timezone=TIMEZONE), args=[reminder_id], id=reminder_id, replace_existing=True)
    if start_scheduler and not scheduler.running:
        scheduler.start()
    return {"id": reminder_id, "text": text, "run_at": run_at.isoformat(), "recurrence": recurrence, "active": 1}

def list_reminders(active_only=True):
    query = "SELECT * FROM reminders"
    if active_only:
        query += " WHERE active = 1"
    query += " ORDER BY run_at"
    with connect_db() as db:
        return [dict(row) for row in db.execute(query)]

def cancel_reminder(reminder_id):
    try:
        scheduler.remove_job(reminder_id)
    except Exception:
        pass
    with connect_db() as db:
        result = db.execute("UPDATE reminders SET active = 0 WHERE id = ?", (reminder_id,))
    if result.rowcount == 0:
        raise ValueError("Reminder not found")
    return {"cancelled": reminder_id}

def start_scheduler():
    if not scheduler.running:
        scheduler.start()
    print("Scheduler running. It stops when the Colab runtime disconnects.")

start_scheduler()

## 5. Safe system status

Only read-only resource information is exposed. No arbitrary shell command or destructive system operation is implemented.

In [ ]:
def system_status():
    memory = psutil.virtual_memory()
    disk = psutil.disk_usage("/")
    return {
        "cpu_percent": psutil.cpu_percent(interval=0.2),
        "ram_percent": memory.percent,
        "ram_available_gb": round(memory.available / 1024**3, 2),
        "disk_free_gb": round(disk.free / 1024**3, 2),
        "scheduler_running": scheduler.running,
        "database": str(DB_PATH.resolve()),
    }

def format_status():
    status = system_status()
    return (f"CPU {status['cpu_percent']}%, RAM {status['ram_percent']}% "
            f"({status['ram_available_gb']} GB available), disk free "
            f"{status['disk_free_gb']} GB, scheduler "
            f"{'running' if status['scheduler_running'] else 'stopped'}.")

## 6. Natural-language command router

This deterministic router is the safe baseline. A Hugging Face or cloud model can be added later to produce validated structured intents, but it should call these same functions rather than execute generated Python.

In [ ]:
def _task_lines(tasks):
    if not tasks:
        return "No pending tasks."
    return "\n".join(f"{task['id']}. [{task['priority']}] {task['title']}" for task in tasks)

def handle_command(command):
    command = _validate_text(command, "Command")
    lower = command.lower()
    with connect_db() as db:
        db.execute("INSERT INTO chat_history(role, content, created_at) VALUES (?, ?, ?)", ("user", command, now_iso()))
    if lower in {"status", "system status", "check system"}:
        response = format_status()
    elif re.match(r"^(show|list) (my )?(pending )?tasks$", lower):
        response = _task_lines(list_tasks())
    elif lower.startswith("add task ") or lower.startswith("create task "):
        title = command.split(" ", 2)[2]
        response = f"Created task {create_task(title)['id']}: {title}"
    elif lower.startswith("complete task "):
        response = f"Completed task {complete_task(int(command.split()[-1]))['id']}."
    elif lower.startswith("delete task "):
        response = f"Deleted task {delete_task(int(command.split()[-1]))['deleted']}."
    elif lower in {"show reminders", "list reminders", "my reminders"}:
        reminders = list_reminders()
        response = "No active reminders." if not reminders else "\n".join(f"{r['id']}: {r['text']} at {r['run_at']}" for r in reminders)
    elif lower.startswith("cancel reminder "):
        response = f"Cancelled reminder {cancel_reminder(command.split()[-1])['cancelled']}."
    else:
        match = re.match(r"remind me to (.+?) at (.+)$", command, re.IGNORECASE)
        if match:
            reminder = create_reminder(match.group(1), match.group(2))
            response = f"Reminder {reminder['id']} set for {reminder['run_at']}: {reminder['text']}"
        else:
            response = "I can manage tasks, reminders, and read-only system status."
    with connect_db() as db:
        db.execute("INSERT INTO chat_history(role, content, created_at) VALUES (?, ?, ?)", ("assistant", response, now_iso()))
    return response

## 7. Test every basic feature

    The tests use unique records so they are safe to run repeatedly in the same Colab session.

In [ ]:
suffix = uuid.uuid4().hex[:6]
task = create_task(f"Test Java study {suffix}", priority="high")
assert task["completed"] == 0
assert get_task(task["id"])["title"].endswith(suffix)
assert update_task(task["id"], priority="low")["priority"] == "low"
assert complete_task(task["id"])["completed"] == 1
assert delete_task(task["id"])["deleted"] == task["id"]

reminder = create_reminder(f"Test reminder {suffix}", "tomorrow at 7:00 PM", start_scheduler=False)
assert any(item["id"] == reminder["id"] for item in list_reminders())
assert cancel_reminder(reminder["id"])["cancelled"] == reminder["id"]

assert "Created task" in handle_command(f"create task Test routed task {suffix}")
assert "pending tasks" in handle_command("show my pending tasks").lower() or "test routed task" in handle_command("show my pending tasks").lower()
assert "CPU" in handle_command("status")
print("All task, reminder, parser, persistence, and system-status tests passed.")

## 8. Simple chat interface

    Run this cell in Colab to get a small chat box and live task/reminder lists. The interface calls the same validated command router.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

chat_output = widgets.Output(layout={"border": "1px solid #ccc", "height": "260px", "overflow_y": "auto"})
command_box = widgets.Text(placeholder="Try: Remind me to study Java at 7 PM")
send_button = widgets.Button(description="Send", button_style="primary")
refresh_button = widgets.Button(description="Refresh lists")
lists_output = widgets.Output()

def refresh_lists(_=None):
    with lists_output:
        clear_output()
        print("PENDING TASKS")
        print(_task_lines(list_tasks()))
        print("\nACTIVE REMINDERS")
        reminders = list_reminders()
        print("No active reminders." if not reminders else "\n".join(f"{r['id']}: {r['text']} at {r['run_at']}" for r in reminders))

def send_command(_=None):
    if not command_box.value.strip():
        return
    command = command_box.value
    command_box.value = ""
    with chat_output:
        print(f"You: {command}")
        try:
            print(f"JARVIS: {handle_command(command)}\n")
        except (ValueError, TypeError) as error:
            print(f"JARVIS: I could not complete that safely: {error}\n")
    refresh_lists()

send_button.on_click(send_command)
command_box.on_submit(send_command)
refresh_button.on_click(refresh_lists)
display(widgets.VBox([chat_output, widgets.HBox([command_box, send_button]), refresh_button, lists_output]))
refresh_lists()


## Try these commands

```text
Create task Finish the Python project
Show my pending tasks
Complete task 1
Remind me to study Java at 7 PM
List reminders
Cancel reminder <id>
Status
```

To stop the scheduler cleanly before restarting notebook code:

```python
scheduler.shutdown(wait=True)
```

For production, the next stages should add authenticated API access, validated tool schemas, a real notification channel, timezone-aware recurring rules, and a persistent worker outside Colab.